In [0]:
%sql
Merge into new_catlog.intellibi_gold.Refinedmonthly_sales as tgt
using new_catlog.intellibi_silver.cleansedmonthly_sales as src
on tgt.sale_id = src.sale_id

-- when record exists but something changed ---> UPDATE (SCD TYPE 1)

when matched and(
    tgt.product <> src.product or
    tgt.category <> src.category or
    tgt.quantity <> src.quantity or
    tgt.price <> src.price or
    tgt.sale_date <> src.sale_date or
    tgt.region <> src.region
) then update set
    tgt.product = src.product,
    tgt.category = src.category,
    tgt.quantity = src.quantity,
    tgt.price = src.price,
    tgt.sale_date = src.sale_date,
    tgt.region = src.region,
    tgt.last_updt_ts = current_timestamp()

-- when record does not exist ---> INSERT

when not matched then insert (
    sale_id,
    product,
    category,
    quantity,
    price,
    sale_date,
    region,
    ingest_load_ts,
    last_updt_ts

) 
values (
    src.sale_id,
    src.product,
    src.category,
    src.quantity,
    src.price,
    src.sale_date,
    src.region,
    current_timestamp(),
    current_timestamp()
)